# 00 · Download qualifying data (2018–2025)

Downloads every qualifying session with [FastF1](https://docs.fastf1.dev) and saves one parquet file per session to Google Drive.

**Rate limits:** FastF1 enforces hard API limits (Jolpica 200 calls/hour, all APIs 500 calls/hour), so eight seasons will not finish in one run. When the limit is hit the run stops cleanly. Wait about an hour and run the download cell again: finished sessions are skipped.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Download (rerun this cell until nothing is left)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}

import fastf1
from f1sim.paths import get_paths
from f1sim.ingest import ingest_seasons

fastf1.set_log_level("WARNING")
paths = get_paths()
summary = ingest_seasons(range(START_YEAR, END_YEAR + 1), paths)

In [ ]:
# @title Check what's on disk
from f1sim.ingest import load_quali

q = load_quali(paths)
sessions = q.drop_duplicates(["Year", "Round"])
coverage = (
    sessions.groupby("Year")
    .agg(Sessions=("Round", "size"))
    .join(sessions.groupby("Year")["QTimeSource"].value_counts().unstack(fill_value=0))
)
coverage